# FIFA World Cup 2026 - Model Training Explanation

This notebook explains the machine learning models used for predicting World Cup 2026 match outcomes.

## Overview

We use an **Ensemble Model** that combines:

1. **ELO Rating System** (40% weight)
2. **Poisson Distribution Model** (60% weight)

---

## Setup

First, let's import the necessary libraries and load our data.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.models.elo_predictor import ELOPredictor
from src.models.poisson_predictor import PoissonPredictor
from src.models.ensemble_predictor import EnsemblePredictor
from src.data.loader import DataLoader

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

## 1. Load Historical Data

We use historical World Cup data from 1930 to 2014 (4,572 matches).

In [ ]:
# Load data
loader = DataLoader()

# Load World Cup matches
matches_df = loader.load_matches(processed=False)

# Check if we have the expected columns
if 'Year' in matches_df.columns:
    print(f"Total matches: {len(matches_df)}")
    print(f"Period: {matches_df['Year'].min()} - {matches_df['Year'].max()}")
else:
    print(f"Total matches: {len(matches_df)}")
    print("Note: Using alternative data format")

print(f"\nFirst few matches:")
matches_df.head()

### Data Exploration

In [ ]:
# Matches per year
plt.figure(figsize=(14, 6))
matches_per_year = matches_df.groupby('Year').size()
plt.bar(matches_per_year.index, matches_per_year.values, color='steelblue')
plt.xlabel('Year')
plt.ylabel('Number of Matches')
plt.title('World Cup Matches per Tournament')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"Average matches per tournament: {matches_per_year.mean():.1f}")

In [ ]:
# Goal distribution
total_goals = matches_df['Home Team Goals'] + matches_df['Away Team Goals']

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.hist(total_goals, bins=range(0, 15), color='coral', edgecolor='black', alpha=0.7)
plt.xlabel('Total Goals per Match')
plt.ylabel('Frequency')
plt.title('Distribution of Goals per Match')

plt.subplot(1, 2, 2)
plt.hist(matches_df['Home Team Goals'], bins=range(0, 12), alpha=0.5, label='Home', color='blue')
plt.hist(matches_df['Away Team Goals'], bins=range(0, 12), alpha=0.5, label='Away', color='red')
plt.xlabel('Goals')
plt.ylabel('Frequency')
plt.title('Home vs Away Goals')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Average goals per match: {total_goals.mean():.2f}")
print(f"Average home goals: {matches_df['Home Team Goals'].mean():.2f}")
print(f"Average away goals: {matches_df['Away Team Goals'].mean():.2f}")

## 2. ELO Rating System

The ELO rating system measures the relative strength of teams based on their match history.

### How it works:

1. **Initial Rating**: All teams start at 1500
2. **K-factor**: 32 (determines how much ratings change)
3. **Home Advantage**: 100 points added to home team
4. **Update Formula**:
   ```
   New Rating = Old Rating + K × (Actual - Expected)
   ```
   Where:
   - Actual = 1 (win), 0.5 (draw), 0 (loss)
   - Expected = 1 / (1 + 10^((Opponent Rating - Your Rating) / 400))

### Training

In [ ]:
# Initialize and train ELO model
elo_model = ELOPredictor(k_factor=32, home_advantage=100)
elo_model.train_on_historical_data(matches_df)

print("✓ ELO model trained")
print(f"\nTeams rated: {len(elo_model.ratings)}")

In [ ]:
# Get top 20 teams by ELO rating
ratings_df = elo_model.get_all_ratings()
top_20 = ratings_df.head(20)

plt.figure(figsize=(12, 8))
plt.barh(range(len(top_20)), top_20['elo_rating'], color='steelblue')
plt.yticks(range(len(top_20)), top_20['team'])
plt.xlabel('ELO Rating')
plt.title('Top 20 Teams by ELO Rating')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 10 Teams:")
print(top_20.head(10).to_string(index=False))

### Example Prediction with ELO

In [ ]:
# Example: Brazil vs Germany
team_a = "Brazil"
team_b = "Germany"

prediction = elo_model.predict_match(team_a, team_b, is_home_a=True)

print(f"\n{team_a} (Home) vs {team_b} (Away)")
print("=" * 50)
print(f"{team_a} Win Probability: {prediction['home_win']:.1%}")
print(f"Draw Probability: {prediction['draw']:.1%}")
print(f"{team_b} Win Probability: {prediction['away_win']:.1%}")

## 3. Poisson Distribution Model

The Poisson model predicts the number of goals each team will score.

### How it works:

1. **Attack Strength**: Goals scored / Average goals
2. **Defense Strength**: Goals conceded / Average goals
3. **Expected Goals**:
   ```
   λ_home = Attack_home × Defense_away × Average_goals
   λ_away = Attack_away × Defense_home × Average_goals
   ```
4. **Probability**: P(X = k) = (λ^k × e^(-λ)) / k!

### Training

In [ ]:
# Initialize and train Poisson model
poisson_model = PoissonPredictor()
poisson_model.train(matches_df)

print("✓ Poisson model trained")
print(f"\nTeams analyzed: {len(poisson_model.team_stats)}")

In [ ]:
# Analyze team strengths
team_stats = []
for team, stats in poisson_model.team_stats.items():
    if team != 'nan':
        team_stats.append({
            'Team': team,
            'Attack': stats['attack_strength'],
            'Defense': stats['defense_strength'],
            'Matches': stats['matches_played']
        })

stats_df = pd.DataFrame(team_stats).sort_values('Attack', ascending=False)

# Plot top attackers and defenders
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

top_attack = stats_df.head(10)
ax1.barh(range(len(top_attack)), top_attack['Attack'], color='red', alpha=0.7)
ax1.set_yticks(range(len(top_attack)))
ax1.set_yticklabels(top_attack['Team'])
ax1.set_xlabel('Attack Strength')
ax1.set_title('Top 10 Attacking Teams')
ax1.invert_yaxis()

top_defense = stats_df.sort_values('Defense').head(10)
ax2.barh(range(len(top_defense)), top_defense['Defense'], color='blue', alpha=0.7)
ax2.set_yticks(range(len(top_defense)))
ax2.set_yticklabels(top_defense['Team'])
ax2.set_xlabel('Defense Strength (lower is better)')
ax2.set_title('Top 10 Defensive Teams')
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

### Example Prediction with Poisson

In [ ]:
# Example: Brazil vs Germany
prediction = poisson_model.predict_match(team_a, team_b, is_home_a=True)

print(f"\n{team_a} (Home) vs {team_b} (Away)")
print("=" * 50)
print(f"Expected Goals: {prediction['expected_goals_home']:.2f} - {prediction['expected_goals_away']:.2f}")
print(f"Most Likely Score: {prediction['most_likely_score'][0]} - {prediction['most_likely_score'][1]}")
print(f"\n{team_a} Win: {prediction['home_win']:.1%}")
print(f"Draw: {prediction['draw']:.1%}")
print(f"{team_b} Win: {prediction['away_win']:.1%}")

## 4. Ensemble Model

The ensemble combines both models for better predictions:

```
Final Prediction = 0.4 × ELO + 0.6 × Poisson
```

### Why Ensemble?

- **ELO**: Good at capturing team strength and momentum
- **Poisson**: Good at predicting specific scores
- **Together**: More robust and accurate predictions

### Training

In [ ]:
# Initialize and train ensemble
ensemble = EnsemblePredictor()
ensemble.train(matches_df)

print("✓ Ensemble model trained")

### Example Prediction with Ensemble

In [ ]:
# Example: Brazil vs Germany
prediction = ensemble.predict_match(team_a, team_b, is_home_a=True)

print(f"\n{team_a} (Home) vs {team_b} (Away)")
print("=" * 50)
print(f"\nWin Probabilities:")
print(f"  {team_a}: {prediction['home_win']:.1%}")
print(f"  Draw: {prediction['draw']:.1%}")
print(f"  {team_b}: {prediction['away_win']:.1%}")
print(f"\nScore Prediction:")
print(f"  Expected: {prediction['expected_goals_home']:.2f} - {prediction['expected_goals_away']:.2f}")
print(f"  Most Likely: {prediction['most_likely_score'][0]} - {prediction['most_likely_score'][1]}")

### Compare All Three Models

In [ ]:
# Compare predictions
elo_pred = elo_model.predict_match(team_a, team_b, is_home_a=True)
poisson_pred = poisson_model.predict_match(team_a, team_b, is_home_a=True)
ensemble_pred = ensemble.predict_match(team_a, team_b, is_home_a=True)

comparison = pd.DataFrame({
    'Model': ['ELO', 'Poisson', 'Ensemble'],
    f'{team_a} Win': [
        f"{elo_pred['home_win']:.1%}",
        f"{poisson_pred['home_win']:.1%}",
        f"{ensemble_pred['home_win']:.1%}"
    ],
    'Draw': [
        f"{elo_pred['draw']:.1%}",
        f"{poisson_pred['draw']:.1%}",
        f"{ensemble_pred['draw']:.1%}"
    ],
    f'{team_b} Win': [
        f"{elo_pred['away_win']:.1%}",
        f"{poisson_pred['away_win']:.1%}",
        f"{ensemble_pred['away_win']:.1%}"
    ]
})

print(f"\nModel Comparison: {team_a} vs {team_b}")
print("=" * 70)
print(comparison.to_string(index=False))

## 5. Model Validation

Let's test the model on some historical matches to see how accurate it is.

In [ ]:
# Sample some recent matches for validation
recent_matches = matches_df[matches_df['Year'] >= 2010].sample(10, random_state=42)

results = []
for _, match in recent_matches.iterrows():
    team_a = match['Home Team Name']
    team_b = match['Away Team Name']
    
    pred = ensemble.predict_match(team_a, team_b, is_home_a=True)
    
    # Determine actual winner
    if match['Home Team Goals'] > match['Away Team Goals']:
        actual = 'Home'
    elif match['Away Team Goals'] > match['Home Team Goals']:
        actual = 'Away'
    else:
        actual = 'Draw'
    
    # Determine predicted winner
    if pred['home_win'] > pred['away_win'] and pred['home_win'] > pred['draw']:
        predicted = 'Home'
    elif pred['away_win'] > pred['home_win'] and pred['away_win'] > pred['draw']:
        predicted = 'Away'
    else:
        predicted = 'Draw'
    
    results.append({
        'Match': f"{team_a} vs {team_b}",
        'Actual': f"{int(match['Home Team Goals'])}-{int(match['Away Team Goals'])}",
        'Predicted': f"{pred['most_likely_score'][0]}-{pred['most_likely_score'][1]}",
        'Winner': actual,
        'Predicted Winner': predicted,
        'Correct': '✓' if actual == predicted else '✗'
    })

results_df = pd.DataFrame(results)
print("\nSample Predictions:")
print("=" * 100)
print(results_df.to_string(index=False))

accuracy = (results_df['Correct'] == '✓').sum() / len(results_df)
print(f"\nAccuracy on sample: {accuracy:.1%}")

## 6. Conclusion

### Model Summary

- **Training Data**: 4,572 World Cup matches (1930-2014)
- **Models**: ELO Rating + Poisson Distribution
- **Ensemble Weight**: 40% ELO + 60% Poisson
- **Output**: Win probabilities and score predictions

### Next Steps

1. Run the dashboard: `streamlit run main.py`
2. View predictions for World Cup 2026
3. Track model accuracy as matches are played

### Limitations

- Historical data only goes to 2014
- Doesn't account for player injuries or form
- Weather and venue conditions not considered
- Team tactics and strategies not modeled

Despite these limitations, the ensemble approach provides robust predictions based on historical performance patterns.